### Extrator - Itaú

In [128]:
import pdfplumber
import pandas as pd
import re
import numpy as np
import json

In [129]:
# para PE e exposição bruta:

# exposição b.: b) Valor Contábil Bruto por Estágios ( pág 52 )
# PE: "c) Perda de Crédito Esperada ( pág 54-55)

# b) Valor Contábil Bruto (Carteira de Crédito) -- 2T25
# b)Valor Contábil Bruto (Carteira de Crédito) -- 1T25
# c)Perda de Crédito Esperada -- 1T25


In [130]:
with pdfplumber.open("Demonstrações Financeiras Itaú - 1T25.pdf") as pdf:
    page = pdf.pages[54]   # página 53 (índice começa em 0)
    text = page.extract_text()

# Remove espaços em branco extras de cada linha ( pré-processamento )
text = "\n".join(line.strip() for line in text.splitlines())

# Define início ( start ) e fim ( end ) do trecho a ser extraído. 
start = text.find("c)Perda de Crédito Esperada")
end = text.find("Consolidado dos 3 Estágios", start)

trecho = text[start:end]
print(trecho)

# divide em linhas para facilitar a criação do dataframe
linhas = [l.strip() for l in trecho.splitlines() if l.strip()]

c)Perda de Crédito Esperada
ReconciliaçãodaperdadecréditoesperadaparaasOperaçõesdeCréditoeArrendamentoMercantilFinanceiro,segregadasporestágios:
01/01/2025
Estágio1 3 S 1 a /1 ld 2 o /2 e 0 m 24 Trans E f s e t r á ê g n i c o ia 2 para Tran E s s f t e á r g ê i n o c 3 ia (1 p ) ara Tran E s s fe tá r g ên io ci 2 ado Tran E s s fe tá r g ên io ci 3 ado WriteOff (Co R n e s v ti e t r u s iç ão ão)/ Sa 3 l 1 d / o 03 fi / n 20 a 2 l 5 em
PessoasFísicas (6.297) 105 7 (1.129) (7) - 933 (6.388)
PessoasJurídicas (2.010) 52 11 (410) (14) - 940 (1.431)
UnidadesExternasAméricaLatina (2.634) 53 5 (518) (339) - 1.677 (1.756)
Total (10.941) 210 23 (2.057) (360) - 3.550 (9.575)
Estágio2 Saldoem Transferênciapara Transferênciapara Transferênciado Transferênciado WriteOff (Constituição)/ Saldofinalem
31/12/2024 Estágio1 Estágio3 Estágio1 Estágio3 Reversão 31/03/2025
PessoasFísicas (5.882) 1.129 1.023 (105) (673) - (4.299) (8.807)
PessoasJurídicas (2.093) 410 428 (52) (406) - (674) (2.387)
Unidade

In [131]:

PRODUTOS = [
    "PessoasFísicas",
    "PessoasJurídicas",
    "Total"
]

def normalizar(txt: str) -> str:
    return txt.replace(" ", "")


def extrair_estagio(linha: str):

    match = re.match(r"\s*Estágio\s*([123])\b", linha)
    if match:
        return f"Estágio{match.group(1)}"
    return None


def eh_produto(linha: str) -> bool:
    linha_n = normalizar(linha)
    return any(linha_n.startswith(normalizar(p)) for p in PRODUTOS)


def reduzir_linha_produto(linha: str):
    partes = linha.split()
    produto = partes[0]

    numeros = [p for p in partes if re.search(r"\d", p)]

    if len(numeros) >= 3:
        return f"{produto} {numeros[2]} {numeros[-1]}"
    
    return linha


def reduzir_texto(texto: str):

    linhas = [l.strip() for l in texto.splitlines() if l.strip()]
    saida = []
    estagio_atual = None

    for linha in linhas:

        # 🔹 Detecta estágio apenas se estiver no INÍCIO da linha
        estagio_detectado = extrair_estagio(linha)
        if estagio_detectado:
            estagio_atual = estagio_detectado
            saida.append(estagio_atual)
            continue

        # 🔹 Ignora linhas híbridas que começam com data
        if re.match(r"\d{2}/\d{2}/\d{4}", linha):
            continue

        # 🔹 Ignora cabeçalhos
        if linha.startswith("Saldo"):
            continue
        
        if linha.startswith("Consolidado"):
            break
        
        if linha.startswith("UnidadesExternas"):
            continue
        

        # 🔹 Só pega produtos se estivermos dentro de um estágio válido
        if estagio_atual and eh_produto(linha):
            saida.append(reduzir_linha_produto(linha))

    return "\n".join(saida)

In [ ]:
texto_reduzido = reduzir_texto(trecho)
print(texto_reduzido)

In [133]:
# para criar a coluna de trimestre

from datetime import datetime

padrao_data = re.compile(r"\b\d{2}/\d{2}/\d{4}\b")
datas = padrao_data.findall(trecho)

print(datas)

def trimestre_from_date(date_str: str) -> str:
    dt = datetime.strptime(date_str, "%d/%m/%Y")
    trimestre_map = {3: "1T", 6: "2T", 9: "3T", 12: "4T"}
    trimestre = trimestre_map.get(dt.month)

    if not trimestre:
        raise ValueError(f"Mês inesperado na data: {date_str}")

    return f"{trimestre}{str(dt.year)[-2:]}"

# pega sempre o terceiro ( data mais recente ) e o último número ( comparação com o ano passado )
def extract_anos(texto: str) -> list[str]:
    datas = re.findall(r"\d{2}/\d{2}/\d{4}", texto)

    if len(datas) < 3:
        raise ValueError("Não foi possível encontrar pelo menos três datas")

    terceira = datas[2]
    ultima = datas[-1]

    return [
        trimestre_from_date(terceira),
        trimestre_from_date(ultima)
    ]

['01/01/2025', '31/12/2024', '31/03/2025', '31/12/2024', '31/03/2025', '31/12/2024', '31/03/2025', '01/01/2024', '31/12/2023', '31/12/2024', '31/12/2023', '31/12/2024', '31/12/2023', '31/12/2024', '31/12/2024']


In [134]:
extract_anos(trecho)

['1T25', '4T24']

In [135]:
# função para reconhecer pontos e virgulas
def conv(x: str):
    x = x.strip()

    if x == "—":
        return None

    # valor contábil negativo (inteiro ou com milhar)
    if re.fullmatch(r"\(\d{1,3}(?:\.\d{3})*\)", x):
        x = "-" + x[1:-1]

    x = x.replace(".", "")
    return float(x)

In [136]:

def parse_linha_estagio(linha):
    partes = linha.split()
    estagio = int(partes[1])

    numeros = partes[2:]
    numeros = [conv(x) for x in numeros]

    return {
        "estagio": estagio,
        "exp_bruta_atual": numeros[0] if len(numeros) > 0 else None,
        "pe_atual": numeros[1] if len(numeros) > 1 else None,
        "exp_bruta_anterior": numeros[2] if len(numeros) > 2 else None,
        "pe_anterior": numeros[3] if len(numeros) > 3 else None,
    }


In [137]:
# separar em blocos para facilitar a visualização:

# tentar resolver o problema sozinho

padrao = re.compile(
    r"(Estágio1.*?)"
    r"(Estágio2.*?)"
    r"(Estágio3.*)",
    re.S | re.M
)

blocos = {"Estágio1": "", "Estágio2": "", "Estágio3": ""}

for m in padrao.finditer(texto_reduzido):
    for i, estagio in enumerate(["Estágio1", "Estágio2", "Estágio3"], start=1):
        if m.group(i):
            blocos[estagio] = m.group(i).strip()

# resultado
for k, v in blocos.items():
    print(f"\n--- {k} ---\n{v}")


--- Estágio1 ---
Estágio1
PessoasFísicas 7 (6.388)
PessoasJurídicas 11 (1.431)
Total 23 (9.575)

--- Estágio2 ---
Estágio2
PessoasFísicas 1.023 (8.807)
PessoasJurídicas 428 (2.387)
Total 1.673 (12.993)

--- Estágio3 ---
Estágio3
PessoasFísicas 673 (14.987)
PessoasJurídicas 406 (6.358)
Total 1.392 (24.686)


In [138]:
def parse_linha_produto(linha: str):

    partes = linha.split()
    produto_raw = partes[0]
    valores = partes[1:]

    valores = [conv(v) for v in valores]

    return produto_raw, valores


In [139]:
anos = extract_anos(trecho)  # ex: ['4T24', '3T25']

MAP_PRODUTO = {
    "PessoasFísicas": "PF",
    "PessoasJurídicas": "PJ",
    "Total": "Total"
}

linhas_df = []

for estagio_nome, bloco in blocos.items():
    estagio_num = int(estagio_nome.replace("Estágio", ""))

    for linha in bloco.splitlines():
        if eh_produto(linha):
            produto_raw, valores = parse_linha_produto(linha)
            produto = MAP_PRODUTO[produto_raw]

            for idx, ano in enumerate(anos):
                linhas_df.append({
                    "ano": ano,
                    "banco": "Itaú",
                    "produto": produto,
                    "Estágio": estagio_num,
                    "Perda Esperada": valores[idx] if idx < len(valores) else None,
                })

df_final = pd.DataFrame(linhas_df)

df_final


,ano,banco,produto,Estágio,Perda Esperada
0,1T25,Itaú,PF,1,7.0
1,4T24,Itaú,PF,1,-6388.0
2,1T25,Itaú,PJ,1,11.0
3,4T24,Itaú,PJ,1,-1431.0
4,1T25,Itaú,Total,1,23.0
5,4T24,Itaú,Total,1,-9575.0
6,1T25,Itaú,PF,2,1023.0
7,4T24,Itaú,PF,2,-8807.0
8,1T25,Itaú,PJ,2,428.0
9,4T24,Itaú,PJ,2,-2387.0


### 📊 Passar para o Excel

In [140]:
import openpyxl
from openpyxl import load_workbook

In [141]:
# Carrega arquivos
dados_bancarios_xlsx = load_workbook("Excel - Dados bancários.xlsx")
itau_sheet = dados_bancarios_xlsx["Itaú"]

# Verifica se existe todos os anos ( se não, adiciona +1 coluna a direita )
anos_df = df_final["ano"].unique()

for ano_df in anos_df:
    
    existe = False
    
    for col in range(1, itau_sheet.max_column + 1):
        if str(itau_sheet.cell(row=5, column=col).value).strip() == str(ano_df):
            existe = True
            break
    
    if not existe:
        nova_coluna = itau_sheet.max_column + 1
        itau_sheet.cell(row=5, column=nova_coluna, value=ano_df)

# Identificação da coluna Tipo
ultima_coluna_df = df_final.columns[-1]

if ultima_coluna_df not in ["Exposição Bruta", "Perda Esperada"]:
    raise ValueError("Última coluna do DataFrame não é métrica válida.")

# Preenchimento dos dados
for _, row in df_final.iterrows():
    
    tipo_df = ultima_coluna_df
    estagio_df = f"Estágio {row['Estágio']}"
    produto_df = row["produto"]
    ano_df = row["ano"]
    valor_df = row[ultima_coluna_df]
    
    linha_encontrada = None
    coluna_encontrada = None
    
    # 🔎 Encontrar linha correta (Tipo + Estágio + Produto)
    for r in range(1, itau_sheet.max_row + 1):
        
        tipo_excel = str(itau_sheet.cell(row=r, column=2).value).strip()
        estagio_excel = str(itau_sheet.cell(row=r, column=3).value).strip()
        produto_excel = str(itau_sheet.cell(row=r, column=4).value).strip()
        
        if (
            tipo_excel == tipo_df and
            estagio_excel == estagio_df and
            produto_excel == produto_df
        ):
            linha_encontrada = r
            break
    
    # 🔎 Encontrar coluna correta (Ano na linha 5)
    for c in range(1, itau_sheet.max_column + 1):
        if str(itau_sheet.cell(row=5, column=c).value).strip() == str(ano_df):
            coluna_encontrada = c
            break
    
    # 🔥 Preencher célula
    if linha_encontrada and coluna_encontrada:
        itau_sheet.cell(
            row=linha_encontrada,
            column=coluna_encontrada,
            value=valor_df
        )

# Salva arquivo
dados_bancarios_xlsx.save("Excel - Dados bancários.xlsx")
